# Approximate nearest neighbors (FAISS, HNSW, IVF)
Approximate nearest-neighbor indexes trade a little recall for a large speedup in vector search. Dense embeddings create the vectors; ANN decides which vectors are worth comparing exactly. Save a copy to Drive to edit.

In [ ]:

import math
import random
import numpy as np
import matplotlib.pyplot as plt

SEED = 16
rng = np.random.default_rng(SEED)
random.seed(SEED)


def l2_normalize(x):
    x = np.asarray(x, dtype=float)
    norm = np.linalg.norm(x, axis=-1, keepdims=True)
    return x / np.maximum(norm, 1e-12)


def minmax(x):
    x = np.asarray(x, dtype=float)
    lo = np.min(x)
    hi = np.max(x)
    span = max(float(hi - lo), 1e-12)
    return (x - lo) / span


def cosine_scores(query, docs):
    q = l2_normalize(np.asarray(query, dtype=float).reshape(1, -1))[0]
    d = l2_normalize(np.asarray(docs, dtype=float))
    return d @ q


def euclidean_distances(points, query):
    points = np.asarray(points, dtype=float)
    query = np.asarray(query, dtype=float)
    return np.linalg.norm(points - query, axis=1)


def rank_desc(scores):
    scores = np.asarray(scores, dtype=float)
    return list(np.argsort(-scores, kind="mergesort"))


def rank_asc(scores):
    scores = np.asarray(scores, dtype=float)
    return list(np.argsort(scores, kind="mergesort"))


def recall_at_1(pred, truth):
    return float(int(pred == truth))


def dcg_at_k(rels, k):
    rels = np.asarray(rels, dtype=float)[:k]
    discounts = np.log2(np.arange(2, len(rels) + 2))
    gains = np.power(2.0, rels) - 1.0
    return float(np.sum(gains / discounts))


def ndcg_at_k(rels, k):
    rels = list(rels)
    ideal = sorted(rels, reverse=True)
    denom = dcg_at_k(ideal, k)
    if denom == 0.0:
        return 0.0
    return dcg_at_k(rels, k) / denom


def average_precision(rels):
    total = 0.0
    hits = 0
    for index, rel in enumerate(rels, start=1):
        if rel > 0:
            hits = hits + 1
            total = total + hits / index
    if hits == 0:
        return 0.0
    return total / hits


def ann_candidates_then_exact(points, query, centroids, assignments, nprobe=1, graph=None, entry=0, ef=3):
    points = np.asarray(points, dtype=float)
    query = np.asarray(query, dtype=float)
    centroids = np.asarray(centroids, dtype=float)
    center_order = rank_asc(euclidean_distances(centroids, query))[:nprobe]
    ivf_candidates = []
    for center_id in center_order:
        members = np.where(np.asarray(assignments) == center_id)[0]
        ivf_candidates.extend(list(members))
    if graph is not None:
        seen = {entry}
        frontier = [entry]
        while frontier and len(seen) < ef:
            current = min(frontier, key=lambda i: np.linalg.norm(points[i] - query))
            frontier.remove(current)
            for neighbor in graph.get(current, []):
                if neighbor not in seen:
                    seen.add(neighbor)
                    frontier.append(neighbor)
        ivf_candidates.extend(sorted(seen))
    candidates = sorted(set(ivf_candidates))
    if not candidates:
        candidates = list(range(len(points)))
    local_distances = euclidean_distances(points[candidates], query)
    best_local = int(np.argmin(local_distances))
    best_index = int(candidates[best_local])
    return best_index, candidates, local_distances


def hybrid_fuse_and_rerank(bm25_scores, dense_scores, rerank_scores=None, alpha=0.5, lam=0.4, normalize=True, top_k=None):
    bm25_scores = np.asarray(bm25_scores, dtype=float)
    dense_scores = np.asarray(dense_scores, dtype=float)
    if normalize:
        sparse = minmax(bm25_scores)
        dense = minmax(dense_scores)
    else:
        sparse = bm25_scores
        dense = dense_scores
    hybrid = alpha * sparse + (1.0 - alpha) * dense
    order = rank_desc(hybrid)
    if top_k is not None:
        order = order[:top_k]
    if rerank_scores is None:
        return hybrid, order, hybrid
    rerank_scores = np.asarray(rerank_scores, dtype=float)
    final = hybrid.copy()
    for doc_id in order:
        final[doc_id] = lam * hybrid[doc_id] + (1.0 - lam) * rerank_scores[doc_id]
    final_order = sorted(order, key=lambda i: -final[i])
    return hybrid, final_order, final


def bi_retrieve_then_cross_rerank(query_vec, doc_vecs, cross_scores, cutoff=3):
    bi_scores = cosine_scores(query_vec, doc_vecs)
    bi_order = rank_desc(bi_scores)
    shortlist = bi_order[:cutoff]
    cross_scores = np.asarray(cross_scores, dtype=float)
    final_order = sorted(shortlist, key=lambda i: -cross_scores[i])
    return bi_scores, shortlist, final_order


def retrieval_metrics(rels, k=5):
    rels = np.asarray(rels, dtype=float)
    top = rels[:k]
    total_relevant = max(float(np.sum(rels > 0)), 1.0)
    precision = float(np.sum(top > 0) / k)
    recall = float(np.sum(top > 0) / total_relevant)
    ndcg = ndcg_at_k(rels, k)
    ap = average_precision(rels)
    return {
        "precision": precision,
        "recall": recall,
        "ndcg": ndcg,
        "ap": ap,
    }


def ann_ladder():
    d1_points = np.array([
        [0.0, 0.0],
        [1.0, 0.0],
        [0.0, 1.0],
        [3.0, 3.0],
        [3.2, 3.1],
        [4.0, 3.0],
    ])
    d1_query = np.array([3.1, 3.0])
    d1_centroids = np.array([
        [0.33, 0.33],
        [3.4, 3.03],
    ])
    d1_assignments = np.array([0, 0, 0, 1, 1, 1])
    d2_points = np.vstack([
        rng.normal([0.0, 0.0], 0.10, size=(8, 2)),
        rng.normal([2.5, 2.5], 0.10, size=(8, 2)),
        rng.normal([5.0, 0.0], 0.10, size=(8, 2)),
    ])
    d2_query = np.array([2.46, 2.52])
    d2_centroids = np.array([
        [0.0, 0.0],
        [2.5, 2.5],
        [5.0, 0.0],
    ])
    d2_assignments = np.repeat(np.arange(3), 8)
    d3_points = np.vstack([
        rng.normal([0.0, 0.0], 0.20, size=(10, 2)),
        rng.normal([1.0, 1.0], 0.25, size=(10, 2)),
        rng.normal([1.4, 1.2], 0.25, size=(10, 2)),
    ])
    d3_query = np.array([1.22, 1.08])
    d3_centroids = np.array([
        [0.0, 0.0],
        [1.0, 1.0],
        [1.4, 1.2],
    ])
    d3_assignments = np.repeat(np.arange(3), 10)
    topic_centers = np.array([
        [1.0, 0.0, 0.2, 0.1],
        [0.1, 1.0, 0.2, 0.1],
        [0.2, 0.1, 1.0, 0.1],
        [0.1, 0.2, 0.1, 1.0],
    ])
    d4_points = np.vstack([center + rng.normal(0.0, 0.06, size=(12, 4)) for center in topic_centers])
    d4_query = topic_centers[2] + np.array([0.02, 0.01, -0.02, 0.00])
    d4_centroids = topic_centers.copy()
    d4_assignments = np.repeat(np.arange(4), 12)
    d5_points = np.vstack([
        rng.normal([0.0, 0.0], 0.15, size=(15, 2)),
        rng.normal([2.0, 2.0], 0.15, size=(15, 2)),
        rng.normal([2.35, 2.05], 0.15, size=(15, 2)),
        rng.normal([4.0, 0.0], 0.15, size=(15, 2)),
    ])
    d5_points[15] = np.array([2.1705, 2.0305])
    d5_query = np.array([2.17, 2.03])
    d5_centroids = np.array([
        [0.0, 0.0],
        [2.18, 2.04],
        [2.0, 2.0],
        [4.0, 0.0],
    ])
    d5_assignments = np.concatenate([
        np.full(15, 0),
        np.full(15, 2),
        np.full(15, 1),
        np.full(15, 3),
    ])
    return [
        {"name": "D1 lesson six points", "points": d1_points, "query": d1_query, "centroids": d1_centroids, "assignments": d1_assignments, "nprobe": 1},
        {"name": "D2 clean clusters", "points": d2_points, "query": d2_query, "centroids": d2_centroids, "assignments": d2_assignments, "nprobe": 1},
        {"name": "D3 overlap and ties", "points": d3_points, "query": d3_query, "centroids": d3_centroids, "assignments": d3_assignments, "nprobe": 2},
        {"name": "D4 20newsgroups-style SVD vectors", "points": d4_points, "query": d4_query, "centroids": d4_centroids, "assignments": d4_assignments, "nprobe": 1},
        {"name": "D5 sparse long-tail clusters", "points": d5_points, "query": d5_query, "centroids": d5_centroids, "assignments": d5_assignments, "nprobe": 1},
    ]


def hybrid_ladder():
    return [
        {"name": "D1 lesson four docs", "bm25": [0.80, 0.20, 0.60, 0.10], "dense": [0.10, 0.90, 0.50, 0.30], "rerank": [0.30, 0.95, 0.55, 0.20], "relevant": {1, 2}, "top_k": 3},
        {"name": "D2 clean lexical plus semantic", "bm25": [0.95, 0.15, 0.65, 0.10, 0.05], "dense": [0.40, 0.88, 0.72, 0.20, 0.10], "rerank": [0.70, 0.92, 0.80, 0.10, 0.05], "relevant": {0, 1, 2}, "top_k": 5},
        {"name": "D3 scale mismatch and synonyms", "bm25": [9.0, 2.0, 6.0, 1.0, 0.5, 0.2], "dense": [0.10, 0.92, 0.55, 0.35, 0.20, 0.10], "rerank": [0.30, 0.98, 0.62, 0.40, 0.15, 0.05], "relevant": {1, 2, 3}, "top_k": 5},
        {"name": "D4 20newsgroups-style query labels", "bm25": [0.75, 0.20, 0.55, 0.48, 0.10, 0.05, 0.35], "dense": [0.55, 0.82, 0.62, 0.57, 0.25, 0.20, 0.52], "rerank": [0.78, 0.85, 0.65, 0.60, 0.20, 0.10, 0.58], "relevant": {0, 1, 2, 3}, "top_k": 5},
        {"name": "D5 exact-id plus vague long tail", "bm25": [12.0, 1.0, 0.5, 0.4, 5.5, 0.2, 0.1, 0.1], "dense": [0.05, 0.90, 0.88, 0.80, 0.20, 0.65, 0.30, 0.10], "rerank": [0.25, 0.95, 0.92, 0.82, 0.35, 0.70, 0.20, 0.05], "relevant": {1, 2, 3, 5}, "top_k": 5},
    ]


def cross_bi_ladder():
    return [
        {"name": "D1 lesson vectors", "query": [1.0, 0.0], "docs": [[0.9, 0.1], [0.7, 0.7], [0.0, 1.0]], "cross": [0.62, 0.88, 0.10], "rels": [2, 3, 0], "cutoff": 3},
        {"name": "D2 clean semantic pairs", "query": [1.0, 0.0, 0.0], "docs": [[0.9, 0.1, 0.0], [0.8, 0.2, 0.1], [0.1, 0.9, 0.0], [0.0, 0.2, 0.8]], "cross": [0.82, 0.90, 0.20, 0.10], "rels": [2, 3, 0, 0], "cutoff": 3},
        {"name": "D3 hard negatives and ties", "query": [1.0, 0.0, 0.0], "docs": [[0.95, 0.05, 0.0], [0.90, 0.10, 0.0], [0.85, 0.15, 0.0], [0.3, 0.9, 0.0], [0.0, 0.1, 0.9]], "cross": [0.55, 0.96, 0.70, 0.15, 0.05], "rels": [1, 3, 2, 0, 0], "cutoff": 4},
        {"name": "D4 lexical pair scorer subset", "query": [0.8, 0.1, 0.1, 0.0], "docs": [[0.7, 0.2, 0.1, 0.0], [0.6, 0.3, 0.1, 0.0], [0.2, 0.7, 0.1, 0.0], [0.1, 0.1, 0.8, 0.0], [0.5, 0.1, 0.3, 0.1], [0.0, 0.0, 0.1, 0.9]], "cross": [0.80, 0.86, 0.30, 0.20, 0.72, 0.05], "rels": [2, 3, 0, 0, 2, 0], "cutoff": 5},
        {"name": "D5 best doc below small cutoff", "query": [1.0, 0.0], "docs": [[0.99, 0.01], [0.96, 0.03], [0.93, 0.04], [0.90, 0.05], [0.75, 0.66], [0.10, 0.99]], "cross": [0.50, 0.52, 0.55, 0.60, 0.99, 0.05], "rels": [1, 1, 1, 2, 3, 0], "cutoff": 3},
    ]


def eval_ladder():
    return [
        {"name": "D1 lesson ranked list", "rels": [1, 0, 1, 1, 0], "slice": "lesson"},
        {"name": "D2 clean multi-query labels", "rels": [1, 1, 1, 0, 0], "slice": "head"},
        {"name": "D3 ties and incomplete judgments", "rels": [0, 1, 1, 0, 1], "slice": "ambiguous"},
        {"name": "D4 query-label subset", "rels": [1, 0, 1, 0, 0], "slice": "medium"},
        {"name": "D5 rare-query long tail", "rels": [0, 0, 0, 1, 1], "slice": "rare"},
    ]


## D1 concept: ANN compares exactly inside a candidate set
Exact Euclidean search returns $\arg\min_i\|x_i-q\|_2$. ANN restricts the expensive scan to $C(q)$ and returns $\arg\min_{i\in C(q)}\|x_i-q\|_2$. The lesson numbers require distances $0.100$, $0.141$, and $0.900$.

In [ ]:

points = np.array([
    [0.0, 0.0],
    [1.0, 0.0],
    [0.0, 1.0],
    [3.0, 3.0],
    [3.2, 3.1],
    [4.0, 3.0],
])
query = np.array([3.1, 3.0])
distances = euclidean_distances(points, query)
exact_order = rank_asc(distances)
exact_top = exact_order[0]
print(np.round(distances, 3))
assert round(float(distances[3]), 3) == 0.100
assert round(float(distances[4]), 3) == 0.141
assert round(float(distances[5]), 3) == 0.900
assert exact_top == 3


## Add IVF and HNSW-style candidate generation
A coarse quantizer sends $q$ to the nearest lists before exact scoring. Here $\|q-c_1\|_2=3.847$ and $\|q-c_2\|_2=0.302$, so one probed list cuts fine comparisons from six to three without losing the true neighbor.

In [ ]:

centroids = np.array([
    [0.33, 0.33],
    [3.4, 3.03],
])
assignments = np.array([0, 0, 0, 1, 1, 1])
center_distances = euclidean_distances(centroids, query)
best, candidates, local_distances = ann_candidates_then_exact(points, query, centroids, assignments, nprobe=1)
hnsw_graph = {
    0: [1, 2],
    1: [0, 3],
    2: [0, 3],
    3: [1, 2, 4, 5],
    4: [3, 5],
    5: [3, 4],
}
hnsw_best, hnsw_candidates, hnsw_distances = ann_candidates_then_exact(points, query, centroids, assignments, nprobe=1, graph=hnsw_graph, entry=0, ef=5)
print("centroid distances", np.round(center_distances, 3))
print("candidates", candidates)
print("ann best", best)
print("hnsw-style candidates", hnsw_candidates)
assert round(float(center_distances[0]), 3) == 3.847
assert round(float(center_distances[1]), 3) == 0.302
assert best == exact_top
assert hnsw_best == exact_top
assert len(candidates) == 3


## Dataset ladder D1→D5
The same ANN routine moves from the six lesson points through clean clusters, overlaps, a 20newsgroups-style vector set, and sparse long-tail clusters.

In [ ]:

rungs = ann_ladder()
for rung in rungs:
    points = rung["points"]
    print(rung["name"], "shape", points.shape, "nprobe", rung["nprobe"])
    print("sample", np.round(points[:3], 3))


## Run the same method across D1→D5
Metric: recall@1 versus exact brute-force search.

In [ ]:

ann_results = []
for rung in rungs:
    points = rung["points"]
    query = rung["query"]
    exact = rank_asc(euclidean_distances(points, query))[0]
    ann, candidates, local_distances = ann_candidates_then_exact(points, query, rung["centroids"], rung["assignments"], nprobe=rung["nprobe"])
    metric = recall_at_1(ann, exact)
    ann_results.append({"name": rung["name"], "exact": exact, "ann": ann, "metric": metric, "candidates": candidates})
for item in ann_results:
    print(f"{item['name']}: exact={item['exact']} ann={item['ann']} recall@1={item['metric']:.3f} candidates={len(item['candidates'])}")


## Results visualization
Left: one candidate-region panel per rung. Right: recall@1 curve as complexity increases.

In [ ]:

fig, axes = plt.subplots(1, 6, figsize=(18, 3))
for ax, rung, result in zip(axes[:5], rungs, ann_results):
    points = rung["points"]
    query = rung["query"]
    candidates = result["candidates"]
    ax.scatter(points[:, 0], points[:, 1], c="lightgray", label="all")
    ax.scatter(points[candidates, 0], points[candidates, 1], c="tab:blue", label="candidates")
    ax.scatter(query[0], query[1], c="black", marker="x", s=80, label="query")
    ax.scatter(points[result["exact"], 0], points[result["exact"], 1], c="tab:green", marker="*", s=100, label="exact")
    ax.set_title(rung["name"].split()[0])
    ax.set_xticks([])
    ax.set_yticks([])
axes[5].plot(range(1, 6), [item["metric"] for item in ann_results], marker="o")
axes[5].set_ylim(-0.05, 1.05)
axes[5].set_xticks(range(1, 6))
axes[5].set_title("recall@1")
axes[5].set_xlabel("rung")
fig.tight_layout()


## Pitfall on D5: using too few probes
A fast IVF lookup that searches the wrong candidate list is just a fast miss. We reproduce the miss with one probe, then increase probes so the true nearest cluster is scored.

In [ ]:

hard = rungs[-1]
points = hard["points"]
query = hard["query"]
exact = rank_asc(euclidean_distances(points, query))[0]
one_probe, one_candidates, one_distances = ann_candidates_then_exact(points, query, hard["centroids"], hard["assignments"], nprobe=1)
two_probe, two_candidates, two_distances = ann_candidates_then_exact(points, query, hard["centroids"], hard["assignments"], nprobe=2)
print("one probe", one_probe, "recall", recall_at_1(one_probe, exact), "candidates", len(one_candidates))
print("two probes", two_probe, "recall", recall_at_1(two_probe, exact), "candidates", len(two_candidates))


## Evaluate it + Practice
- Metric: recall@1 against brute-force exact search, plus candidate-count as a no-skill speed baseline.
- Sanity check: if $C(q)$ contains every point, ANN must match exact search.
- Ablation: set `nprobe=1` on D5 and recall should drop when the true cluster is skipped.
- Failure signal: latency improves but recall@1 collapses on long-tail queries.

Practice: change D5 to `nprobe=3` and plot recall versus candidate count.

Practice: replace Euclidean distance with cosine after `l2_normalize` and compare the top neighbor.